# **RendaFlex AI**
Assistente Financeiro para pessoas com renda variável.

**O objetivo do projeto** é Classificar transações, Avaliar perfil financeiro e simular decisões.

O dataset utilizado é Synthetic Bank Transactions, do kaggle.com, contendo:

* Clientes – informações básicas sobre os usuários do banco.
* Categorias – categorias padrão de transações utilizadas por muitos bancos em todo o mundo.
* Transações – o núcleo do nosso conjunto de dados, contendo informações básicas sobre as transações, como a conta da contraparte (segunda conta envolvida na transação), a categoria, o valor, entre outros detalhes.
* Assinaturas – informações sobre assinaturas, ou seja, transações realizadas automaticamente.

## 1. Bibliotecas

In [ ]:
# Bibliotecas de sempre pra manipulação e visualização de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import time
import joblib

# Tradutor, pra dar conta das colunas em russo
!pip install deep-translator --quiet
from deep_translator import GoogleTranslator

# Bibliotecas de ML: uma pro texto (classificação de despesas), outra pro tabular (perfil financeiro)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)


## 2. Carregando os dados

Nada de tratamento ainda, só carregando as 4 tabelas do dataset pra ver como elas chegam.

In [ ]:
df_clients = pd.read_csv('clients.csv')
df_categories = pd.read_csv('categories.csv')
df_transactions = pd.read_csv('transactions.csv')
df_subscriptions = pd.read_csv('subscriptions.csv')

for nome, df in [('clients', df_clients), ('categories', df_categories),
                  ('transactions', df_transactions), ('subscriptions', df_subscriptions)]:
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")


In [ ]:
display(df_clients.head(3))
display(df_categories.head(3))
display(df_transactions.head(3))
display(df_subscriptions.head(3))


## 3. Traduzindo do russo pro português

O dataset original vem com nomes de categoria, empresa e alguns textos em russo. Só que não faz sentido
mandar cada uma das quase 929 mil transações pro tradutor — além de demorar uma eternidade, a gente ia
esbarrar em limite de requisição da API. A saída é traduzir só os valores ÚNICOS das tabelas pequenas
(categorias, nome de empresa em assinaturas) e depois usar esse "dicionário" pra bater com as tabelas grandes.

A função abaixo guarda num cache tudo que já foi traduzido (não traduz a mesma palavra duas vezes) e tenta
de novo se a tradução falhar (às vezes o serviço engasga).

In [ ]:
tradutor = GoogleTranslator(source='ru', target='pt')
_cache_traducao = {}

def traduzir(texto, tentativas=3):
    """Traduz um texto do russo pro português. Guarda em cache pra não pedir a mesma
    tradução duas vezes, e tenta de novo algumas vezes caso o serviço falhe."""
    if pd.isna(texto) or str(texto).strip() == '':
        return texto
    texto = str(texto)
    if texto in _cache_traducao:
        return _cache_traducao[texto]
    for tentativa in range(tentativas):
        try:
            traduzido = tradutor.translate(texto)
            _cache_traducao[texto] = traduzido
            return traduzido
        except Exception:
            time.sleep(1)
    # se não conseguir traduzir depois de todas as tentativas, mantém o texto original
    _cache_traducao[texto] = texto
    return texto

def traduzir_coluna(df, coluna):
    """Traduz só os valores únicos de uma coluna e depois aplica o de-para na coluna inteira.
    Bem mais rápido do que traduzir linha a linha quando a tabela é grande."""
    valores_unicos = df[coluna].dropna().unique()
    de_para = {v: traduzir(v) for v in valores_unicos}
    return df[coluna].map(de_para).fillna(df[coluna])


In [ ]:
# Categorias é uma tabela pequena (29 linhas), então traduzir é rápido e direto
df_categories['name'] = traduzir_coluna(df_categories, 'name')
df_categories['description'] = traduzir_coluna(df_categories, 'description')

df_categories = df_categories.rename(columns={
    'id': 'id_categoria',
    'name': 'categoria',
    'description': 'descricao',
    'mcc-code': 'codigo_mcc'
})

display(df_categories.head(29))


In [ ]:
# Aqui só traduzimos os nomes de empresa que aparecem nas assinaturas (bem menos linhas que transactions)
df_subscriptions['product_company'] = traduzir_coluna(df_subscriptions, 'product_company')

df_subscriptions = df_subscriptions.rename(columns={
    'client_id': 'id_cliente',
    'product_category': 'id_categoria',
    'product_company': 'empresa',
    'amount': 'valor',
    'date_start': 'data_inicio',
    'date_end': 'data_fim'
})

display(df_subscriptions.head(10))


## 4. Organizando os dados dos clientes

Aqui a gente decide o que fica e o que sai. Nome, endereço, telefone, e-mail e local de trabalho são dados
sintéticos (fake) e, mesmo traduzidos, não ajudam o modelo a aprender nada — são praticamente um identificador
único por pessoa, não um padrão que se repete. Por isso saem da base de treino (se um dia vocês quiserem
extrair a cidade do endereço pra usar como variável de região, dá pra voltar aqui e mudar).

O resto (renda, despesas, crédito, depósito) vem como texto e tem espaço em branco no lugar de vazio, então
convertendo pra número. E calculamos a idade a partir da data de nascimento, que pode pesar no perfil.

In [ ]:
df_clients_clean = df_clients.rename(columns={
    'id': 'id_cliente',
    'birthdate': 'data_nascimento',
    'registration_date': 'data_cadastro',
    'gender': 'genero',
    'income': 'renda_cadastro',
    'expenses': 'despesas_cadastro',
    'credit': 'possui_credito',
    'deposit': 'possui_deposito'
})

# fora os dados pessoais, não usamos isso no modelo
df_clients_clean = df_clients_clean.drop(columns=['fullname', 'address', 'phone_number', 'email', 'workplace'])

for col in ['renda_cadastro', 'despesas_cadastro', 'possui_credito', 'possui_deposito']:
    df_clients_clean[col] = pd.to_numeric(df_clients_clean[col], errors='coerce')

df_clients_clean['data_nascimento'] = pd.to_datetime(df_clients_clean['data_nascimento'], errors='coerce')
df_clients_clean['data_cadastro'] = pd.to_datetime(df_clients_clean['data_cadastro'], errors='coerce')
df_clients_clean['idade'] = (pd.Timestamp('2020-12-31') - df_clients_clean['data_nascimento']).dt.days // 365

display(df_clients_clean.head(5))
print(df_clients_clean.isna().mean().round(3))  # proporção de dados faltando em cada coluna


## 5. Organizando as transações

Essa é a tabela principal pra tese de renda variável. Passos:

1. Transformar a data em algo que dê pra agrupar por mês.
2. Trazer o nome da categoria (já traduzido) pra cada transação.
3. Separar o que é renda (Positive) do que é despesa (Negative).

In [ ]:
df_transactions_clean = df_transactions.rename(columns={
    'client_id': 'id_cliente',
    'product_category': 'id_categoria',
    'product_company': 'empresa',
    'subtype': 'subtipo',
    'amount': 'valor',
    'date': 'data',
    'transaction_type': 'tipo_transacao'
}).drop(columns=[c for c in ['Unnamed: 0'] if c in df_transactions.columns])

df_transactions_clean['data'] = pd.to_datetime(df_transactions_clean['data'])
df_transactions_clean['ano_mes'] = df_transactions_clean['data'].dt.to_period('M')

df_transactions_clean = df_transactions_clean.merge(
    df_categories[['id_categoria', 'categoria', 'codigo_mcc']],
    on='id_categoria', how='left'
)

print(df_transactions_clean['tipo_transacao'].value_counts())
display(df_transactions_clean.head(5))


## 6. Agrupando por cliente e mês

Essa é a peça central do projeto. Pra cada cliente, em cada mês do ano, somamos quanto entrou (renda) e
quanto saiu (despesa), e calculamos o saldo. É essa janela cliente-mês que vai deixar a gente enxergar se
a renda de alguém varia muito de um mês pro outro — o que é exatamente o comportamento de quem vive de
bico, freelance ou trabalho autônomo.

In [ ]:
renda_mensal = (
    df_transactions_clean[df_transactions_clean['tipo_transacao'] == 'Positive']
    .groupby(['id_cliente', 'ano_mes'])['valor']
    .agg(renda_mes='sum', qtd_transacoes_renda='count')
    .reset_index()
)

despesa_mensal = (
    df_transactions_clean[df_transactions_clean['tipo_transacao'] == 'Negative']
    .groupby(['id_cliente', 'ano_mes'])['valor']
    .agg(despesa_mes=lambda x: x.abs().sum(), qtd_transacoes_despesa='count')
    .reset_index()
)

user_month = renda_mensal.merge(despesa_mensal, on=['id_cliente', 'ano_mes'], how='outer').fillna(0)
user_month['saldo_mes'] = user_month['renda_mes'] - user_month['despesa_mes']

display(user_month.head(10))
print(f"Total de linhas cliente-mês: {len(user_month)}")
print(f"Clientes únicos: {user_month['id_cliente'].nunique()}")


## 7. Medindo a instabilidade da renda

Pra cada cliente, olhamos a renda mensal ao longo do ano e calculamos a média, o desvio padrão e o
coeficiente de variação (desvio padrão dividido pela média). Quanto maior esse número, mais a renda
oscila de mês pra mês — é o retrato numérico de quem tem renda variável.

Só uma ressalva: se o cliente só tem renda registrada em 1 ou 2 meses, esse cálculo não é confiável
(não dá pra falar de "variação" com tão pouco dado). Por isso marcamos quem tem pelo menos 3 meses.

In [ ]:
perfil_renda = (
    user_month[user_month['renda_mes'] > 0]
    .groupby('id_cliente')['renda_mes']
    .agg(renda_media='mean', renda_desvio='std', meses_com_renda='count')
    .reset_index()
)

perfil_renda['renda_desvio'] = perfil_renda['renda_desvio'].fillna(0)
perfil_renda['coef_variacao_renda'] = (perfil_renda['renda_desvio'] / perfil_renda['renda_media']).fillna(0)

MIN_MESES = 3
perfil_renda['renda_confiavel'] = perfil_renda['meses_com_renda'] >= MIN_MESES

print(f"Clientes com renda confiável (>= {MIN_MESES} meses): "
      f"{perfil_renda['renda_confiavel'].sum()} de {len(perfil_renda)}")

display(perfil_renda.sort_values('coef_variacao_renda', ascending=False).head(10))


## 8. Separando despesa fixa de despesa variável

Cruzamos as assinaturas (débitos automáticos, recorrências) com os clientes pra saber quanto de gasto fixo
cada um carrega por mês. Isso importa porque duas pessoas podem ter a mesma renda instável, mas uma delas
tem um monte de compromisso fixo (aluguel, assinatura, financiamento) e a outra não — o risco financeiro
de cada uma é bem diferente.

In [ ]:
df_subscriptions['data_inicio'] = pd.to_datetime(df_subscriptions['data_inicio'], errors='coerce')
df_subscriptions['data_fim'] = pd.to_datetime(df_subscriptions['data_fim'], errors='coerce')

despesa_fixa_mensal = (
    df_subscriptions.groupby('id_cliente')['valor']
    .sum()
    .reset_index()
    .rename(columns={'valor': 'despesa_fixa_total'})
)

despesa_fixa_mensal['despesa_fixa_mensal_media'] = despesa_fixa_mensal['despesa_fixa_total'] / 12

display(despesa_fixa_mensal.head(10))


## 9. Montando a tabela final de atributos por cliente

Aqui juntamos tudo que calculamos até agora numa única tabela: renda média, instabilidade de renda,
despesa fixa, despesa total e saldo médio. É essa tabela que vira a entrada do modelo de perfil financeiro.
Ficamos só com os clientes que passaram no filtro de "renda confiável" da etapa 7.

In [ ]:
despesa_media_mensal = (
    user_month.groupby('id_cliente')['despesa_mes']
    .mean()
    .reset_index()
    .rename(columns={'despesa_mes': 'despesa_media_mensal'})
)

saldo_medio_mensal = (
    user_month.groupby('id_cliente')['saldo_mes']
    .mean()
    .reset_index()
    .rename(columns={'saldo_mes': 'saldo_medio_mensal'})
)

features_cliente = (
    perfil_renda
    .merge(despesa_media_mensal, on='id_cliente', how='left')
    .merge(despesa_fixa_mensal, on='id_cliente', how='left')
    .merge(saldo_medio_mensal, on='id_cliente', how='left')
    .merge(df_clients_clean[['id_cliente', 'idade', 'genero']], on='id_cliente', how='left')
)

features_cliente['despesa_fixa_mensal_media'] = features_cliente['despesa_fixa_mensal_media'].fillna(0)
features_cliente['comprometimento_fixo'] = (
    features_cliente['despesa_fixa_mensal_media'] / features_cliente['renda_media']
).replace([np.inf, -np.inf], np.nan)

features_cliente = features_cliente[features_cliente['renda_confiavel']].reset_index(drop=True)

display(features_cliente.head(10))
print(f"Total de clientes na base final: {len(features_cliente)}")


## 10. Definindo o rótulo do perfil financeiro

O dataset não vem com a etiqueta "Saudável / Em observação / Em risco" pronta — quem inventa essa regra
somos nós. A ideia abaixo é um ponto de partida: quem tem saldo negativo ou compromete demais a renda com
gasto fixo entra em risco; quem tem renda muito instável ou compromete bastante (mas não tanto) fica em
observação; o resto é considerado saudável.

**Importante:** essa regra precisa ser validada e ajustada com o time antes da entrega final — os números
de corte (0.7, 0.4) são um chute inicial baseado em bom senso, não uma verdade absoluta.

In [ ]:
def classificar_perfil(row):
    if row['saldo_medio_mensal'] < 0 or row['comprometimento_fixo'] > 0.7:
        return 'Em risco'
    elif row['coef_variacao_renda'] > 0.4 or row['comprometimento_fixo'] > 0.4:
        return 'Em observação'
    else:
        return 'Saudável'

features_cliente['perfil_financeiro'] = features_cliente.apply(classificar_perfil, axis=1)

print(features_cliente['perfil_financeiro'].value_counts())
px.histogram(features_cliente, x='coef_variacao_renda', color='perfil_financeiro',
             title='Distribuição do coeficiente de variação de renda por perfil')


## 11. Conferindo se a regra faz sentido

Antes de treinar qualquer modelo em cima desse rótulo, vale olhar visualmente se ele separa bem os grupos.
Se os perfis estivessem todos misturados nos gráficos abaixo, seria sinal de que a regra da etapa 10
precisa de ajuste antes de seguir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=features_cliente, x='perfil_financeiro', y='coef_variacao_renda', ax=axes[0])
axes[0].set_title('Instabilidade de renda por perfil')

sns.boxplot(data=features_cliente, x='perfil_financeiro', y='comprometimento_fixo', ax=axes[1])
axes[1].set_title('Comprometimento com despesa fixa por perfil')

plt.tight_layout()
plt.show()


## 12. Classificando a descrição da transação em categoria

Essa etapa é separada das anteriores porque resolve um problema diferente: dado o texto solto que descreve
uma transação (ex: "Supermercado", "Combustível", "Netflix"), prever automaticamente em qual das categorias
do hackathon ela se encaixa (Alimentação, Transporte, Saúde, Moradia, Educação, Lazer, Serviços, Outras).

O dataset Synthetic Bank Transactions não tem texto livre de transação (ele já vem com a categoria como
número/ID). Por isso, montamos aqui uma base de exemplos de descrição -> categoria pra treinar esse
classificador. Na versão final do time, essa base deve ser trocada/ampliada pelo dataset de descrições reais
que a equipe decidir usar (ex: Financial-Control-Dataset) — o código abaixo já funciona do mesmo jeito,
só muda a fonte dos dados.

In [ ]:
# Exemplos de descrição de transação -> categoria do hackathon.
# Isso é um ponto de partida pra fins de demonstração do pipeline; o ideal é substituir/somar
# esses exemplos por uma base maior e mais real assim que o time definir a fonte final.
exemplos_descricao = pd.DataFrame([
    ("Supermercado Extra", "Alimentação"),
    ("iFood pedido restaurante", "Alimentação"),
    ("Feira livre", "Alimentação"),
    ("Padaria do bairro", "Alimentação"),
    ("Posto de gasolina", "Transporte"),
    ("Uber viagem centro", "Transporte"),
    ("Passagem de ônibus", "Transporte"),
    ("Estacionamento shopping", "Transporte"),
    ("Farmácia remédio", "Saúde"),
    ("Consulta médica particular", "Saúde"),
    ("Plano de saúde mensalidade", "Saúde"),
    ("Academia mensalidade", "Saúde"),
    ("Aluguel apartamento", "Moradia"),
    ("Conta de luz", "Moradia"),
    ("Conta de água", "Moradia"),
    ("Condomínio mensal", "Moradia"),
    ("Mensalidade faculdade", "Educação"),
    ("Curso online plataforma", "Educação"),
    ("Material escolar", "Educação"),
    ("Livro técnico", "Educação"),
    ("Netflix assinatura", "Lazer"),
    ("Cinema ingresso", "Lazer"),
    ("Show de música", "Lazer"),
    ("Spotify assinatura", "Lazer"),
    ("Salão de beleza", "Serviços"),
    ("Conserto celular", "Serviços"),
    ("Assinatura software", "Serviços"),
    ("Plano de celular", "Serviços"),
    ("Transferência entre contas", "Outras"),
    ("Saque em caixa eletrônico", "Outras"),
], columns=["descricao", "categoria"])

display(exemplos_descricao)


In [ ]:
# Separando treino e teste da base de descrições
X_texto = exemplos_descricao['descricao']
y_texto = exemplos_descricao['categoria']

X_texto_treino, X_texto_teste, y_texto_treino, y_texto_teste = train_test_split(
    X_texto, y_texto, test_size=0.25, random_state=42, stratify=y_texto
)

# TF-IDF transforma o texto em números que o modelo consegue entender,
# dando mais peso pra palavras que aparecem pouco (e por isso ajudam a diferenciar categorias)
vetorizador_tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
X_texto_treino_vet = vetorizador_tfidf.fit_transform(X_texto_treino)
X_texto_teste_vet = vetorizador_tfidf.transform(X_texto_teste)

# Naive Bayes é um ponto de partida clássico e rápido pra classificação de texto
modelo_categoria = MultinomialNB()
modelo_categoria.fit(X_texto_treino_vet, y_texto_treino)

y_texto_pred = modelo_categoria.predict(X_texto_teste_vet)

print("Acurácia:", accuracy_score(y_texto_teste, y_texto_pred))
print()
print(classification_report(y_texto_teste, y_texto_pred, zero_division=0))


## 13. Treinando o modelo de perfil financeiro

Agora a parte tabular: usar as colunas que calculamos (renda média, instabilidade de renda, comprometimento
com despesa fixa, despesa média, idade) pra prever o perfil financeiro (Saudável / Em observação / Em risco).

Usamos RandomForest porque lida bem com esse tipo de dado tabular, é rápido de treinar e ainda dá pra
explicar quais variáveis pesaram mais na decisão (útil pra explicar o resultado pro usuário final).

In [ ]:
colunas_features = [
    'renda_media', 'coef_variacao_renda', 'despesa_media_mensal',
    'comprometimento_fixo', 'saldo_medio_mensal', 'idade'
]

base_modelo = features_cliente.dropna(subset=colunas_features + ['perfil_financeiro']).copy()

X_perfil = base_modelo[colunas_features]
y_perfil = base_modelo['perfil_financeiro']

X_perfil_treino, X_perfil_teste, y_perfil_treino, y_perfil_teste = train_test_split(
    X_perfil, y_perfil, test_size=0.2, random_state=42, stratify=y_perfil
)

modelo_perfil = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
modelo_perfil.fit(X_perfil_treino, y_perfil_treino)

y_perfil_pred = modelo_perfil.predict(X_perfil_teste)

print("Acurácia:", accuracy_score(y_perfil_teste, y_perfil_pred))
print("F1 (média ponderada):", f1_score(y_perfil_teste, y_perfil_pred, average='weighted'))
print()
print(classification_report(y_perfil_teste, y_perfil_pred, zero_division=0))


In [ ]:
# Matriz de confusão: mostra onde o modelo mais confunde um perfil com outro
matriz = confusion_matrix(y_perfil_teste, y_perfil_pred, labels=modelo_perfil.classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=modelo_perfil.classes_, yticklabels=modelo_perfil.classes_)
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.title('Matriz de confusão - perfil financeiro')
plt.show()

# Quais variáveis mais pesaram na decisão do modelo
importancias = pd.Series(modelo_perfil.feature_importances_, index=colunas_features).sort_values(ascending=False)
print(importancias)


## 14. Salvando os modelos treinados

Serializamos os dois modelos (categoria de despesa e perfil financeiro) junto com o vetorizador TF-IDF,
pra não precisar retreinar nada toda vez que a API for chamada. É esse arquivo que o back-end vai carregar.

In [ ]:
joblib.dump(modelo_categoria, 'modelo_categoria_despesa.pkl')
joblib.dump(vetorizador_tfidf, 'vetorizador_tfidf_categoria.pkl')
joblib.dump(modelo_perfil, 'modelo_perfil_financeiro.pkl')

features_cliente.to_csv('features_cliente_rendaflex.csv', index=False)
df_categories.to_csv('categorias_traduzidas.csv', index=False)
user_month.to_csv('user_month_rendaflex.csv', index=False)

print("Modelos e tabelas salvos com sucesso.")


## 15. Função de inferência (o que a API vai chamar)

Essa é a função que junta tudo: recebe os dados financeiros do usuário no mesmo formato do exemplo do
edital, classifica cada transação numa categoria, calcula um perfil financeiro simplificado e devolve
algumas recomendações. É essa função (ou a lógica dela) que o time de back-end vai empacotar dentro da API.

Como aqui a gente só tem os dados de um único momento (não o histórico de 12 meses que usamos pra treinar
o modelo de perfil), calculamos um coeficiente de variação aproximado a partir do nível de endividamento e
da frequência de poupança informados na entrada — é uma adaptação pra deixar a função utilizável com o
formato de entrada que o edital pede, mesmo sem o histórico completo do cliente.

In [ ]:
def analisar_financas(dados_entrada):
    """
    Recebe um dicionário no formato do endpoint POST /analise-financeira e devolve
    o resumo de gastos, o perfil financeiro e recomendações, em formato de dicionário
    (pronto pra virar JSON).
    """
    transacoes = dados_entrada.get('transacoes', [])
    renda_mensal = dados_entrada.get('renda_mensal', 0)
    nivel_endividamento = dados_entrada.get('nivel_endividamento', 0)
    frequencia_poupanca = dados_entrada.get('frequencia_poupanca', 'Baixa')

    # Classifica cada transação e soma os gastos por categoria
    descricoes = [t['descricao'] for t in transacoes]
    valores = [t['valor'] for t in transacoes]

    if descricoes:
        vetor = vetorizador_tfidf.transform(descricoes)
        categorias_previstas = modelo_categoria.predict(vetor)
    else:
        categorias_previstas = []

    resumo_gastos = {}
    for categoria, valor in zip(categorias_previstas, valores):
        chave = categoria.lower().replace('ç', 'c').replace('ã', 'a')
        resumo_gastos[chave] = resumo_gastos.get(chave, 0) + valor

    despesa_total = sum(valores)

    # Como não temos o histórico de 12 meses aqui (só o instante informado na requisição),
    # aproximamos a instabilidade de renda a partir da frequência de poupança informada
    mapa_frequencia = {'Alta': 0.1, 'Media': 0.3, 'Baixa': 0.6}
    coef_variacao_aprox = mapa_frequencia.get(frequencia_poupanca, 0.3)

    comprometimento = despesa_total / renda_mensal if renda_mensal > 0 else 1
    saldo_estimado = renda_mensal - despesa_total

    entrada_modelo = pd.DataFrame([{
        'renda_media': renda_mensal,
        'coef_variacao_renda': coef_variacao_aprox,
        'despesa_media_mensal': despesa_total,
        'comprometimento_fixo': min(comprometimento, 1),
        'saldo_medio_mensal': saldo_estimado,
        'idade': dados_entrada.get('idade', 35)  # valor neutro caso não venha na entrada
    }])

    perfil_previsto = modelo_perfil.predict(entrada_modelo)[0]
    probabilidade = modelo_perfil.predict_proba(entrada_modelo).max()

    # Recomendações simples, baseadas em regras (não precisa de ML pra isso)
    recomendacoes = []
    if nivel_endividamento > 30:
        recomendacoes.append("Priorizar redução do nível de endividamento")
    if comprometimento > 0.6:
        recomendacoes.append("Rever gastos recorrentes, o comprometimento da renda está alto")
    if frequencia_poupanca == 'Baixa':
        recomendacoes.append("Aumentar a frequência de poupança mensal")
    if not recomendacoes:
        recomendacoes.append("Manter os hábitos financeiros atuais")

    return {
        "perfil_financeiro": perfil_previsto,
        "probabilidade": round(float(probabilidade), 2),
        "resumo_gastos": resumo_gastos,
        "recomendacoes": recomendacoes
    }


## 16. Testando com exemplos reais

O edital pede pelo menos 3 exemplos reais de uso. Aqui testamos a função acima com 3 perfis diferentes
de usuário (renda estável, renda variável moderada, renda variável com aperto financeiro), só pra garantir
que ela responde no formato esperado antes de entregar pro back-end.

In [ ]:
exemplo_1 = {
    "renda_mensal": 4500,
    "nivel_endividamento": 25,
    "frequencia_poupanca": "Media",
    "transacoes": [
        {"descricao": "Supermercado Extra", "valor": 420},
        {"descricao": "Posto de gasolina", "valor": 300},
        {"descricao": "Netflix assinatura", "valor": 40}
    ]
}

exemplo_2 = {
    "renda_mensal": 2800,
    "nivel_endividamento": 55,
    "frequencia_poupanca": "Baixa",
    "transacoes": [
        {"descricao": "Aluguel apartamento", "valor": 1200},
        {"descricao": "Farmácia remédio", "valor": 150},
        {"descricao": "iFood pedido restaurante", "valor": 90}
    ]
}

exemplo_3 = {
    "renda_mensal": 7200,
    "nivel_endividamento": 10,
    "frequencia_poupanca": "Alta",
    "transacoes": [
        {"descricao": "Curso online plataforma", "valor": 200},
        {"descricao": "Academia mensalidade", "valor": 150},
        {"descricao": "Show de música", "valor": 250}
    ]
}

for i, exemplo in enumerate([exemplo_1, exemplo_2, exemplo_3], start=1):
    print(f"--- Exemplo {i} ---")
    resultado = analisar_financas(exemplo)
    print(resultado)
    print()


## 17. Próximos passos

- Validar com o time a regra de rotulagem do perfil financeiro (etapa 10) — ela é o ponto mais sensível
  de todo o pipeline, porque o modelo de perfil aprende em cima dela.
- Trocar a base de exemplos de descrição da etapa 12 por uma base maior e real, assim que o time decidir
  qual fonte usar.
- Entregar pro back-end: os arquivos `.pkl` gerados na etapa 14, a função `analisar_financas` da etapa 15
  (isso vira um `inference.py`), e os 3 exemplos testados na etapa 16.
- Entregar pro front-end: apenas os JSONs de saída dos exemplos da etapa 16, sem precisar dos dados brutos
  nem do notebook.
- Se sobrar tempo: dashboard em HTML, explicabilidade do modelo, containerização com Docker — tudo isso
  é opcional segundo o edital, então só entra depois do MVP estar de pé.
